## 1. Extract  
### Data Sources Overview

The following datasets were extracted using the files provided.

1. df_crashes_main  
Source: bitre_fatal_crashes_dec2024.xlsx, Sheet 1  
Content: Contains detailed records of fatal crashes in Australia, including crash ID, state, time, crash type, vehicle involvement, speed limit, and location data.

2. df_fatalities  
Source: bitre_fatalities_dec2024.xlsx, Sheet 1  
Content: Includes individual-level data on fatalities in crashes. Each row represents a person, with demographic information and spatial/temporal details that link to the crash records.

3. df_population_remote  
Source: Population estimates by LGA and other areas.xlsx, Sheet 3  
Content: Contains yearly population estimates by remoteness area category.  
Reason for inclusion: The Remoteness Area values correspond directly with the National Remoteness Areas field in the crash and fatality datasets, enabling population-adjusted analysis.

4. df_population_lga
Source: Population estimates by LGA and other areas.xlsx, Sheet 1
Content: Contains yearly population estimates by Local Government Area (LGA).
Reason for inclusion: The LGA names correspond with National LGA Name in both df_crashes_main and df_fatalities.

5. df_dwellings  
Source: LGA (count of dwellings).csv  
Content: Provides 2021 Census data on the number of dwellings per LGA (Local Government Area).  
Reason for inclusion: This dataset can be directly matched with National LGA Name in both df_crashes_main and df_fatalities, allowing analysis based on residential density. In addition to population data, dwelling data is included because it provides complementary insights—some LGAs may have many high-rise apartments while others consist mostly of detached houses. The number and type of dwellings help better understand the built environment and its potential influence on crash patterns.

### Excluded Datasheets

Sheet 2 from bitre_fatal_crashes_dec2024.xlsx  
Sheet 2 from bitre_fatalities_dec2024.xlsx  

These sheets contain daily aggregated counts of fatal crashes and fatalities. They are excluded because df_crashes_main and df_fatalities only contain month and day of week, without specific dates, so these daily counts cannot be matched back to individual records. Since we are already working with individual-level data, these summaries do not add much additional value and offer limited insight on their own. 

In [1]:
import sys
!{sys.executable} -m pip install pandas
!{sys.executable} -m pip install openpyxl

DEPRECATION: Configuring installation scheme with distutils config files is deprecated and will no longer work in the near future. If you are using a Homebrew or Linuxbrew Python, please see discussion at https://github.com/Homebrew/homebrew-core/issues/76621
DEPRECATION: Configuring installation scheme with distutils config files is deprecated and will no longer work in the near future. If you are using a Homebrew or Linuxbrew Python, please see discussion at https://github.com/Homebrew/homebrew-core/issues/76621

[notice] A new release of pip is available: 24.1.2 -> 25.0.1
[notice] To update, run: python3.9 -m pip install --upgrade pip
DEPRECATION: Configuring installation scheme with distutils config files is deprecated and will no longer work in the near future. If you are using a Homebrew or Linuxbrew Python, please see discussion at https://github.com/Homebrew/homebrew-core/issues/76621
DEPRECATION: Configuring installation scheme with distutils config files is deprecated and wil

In [2]:
import pandas as pd

# Set the base path
base_path = '/Users/lunaxu/Downloads/'

# File 1: bitre_fatal_crashes_dec2024.xlsx
crashes_file = base_path + 'bitre_fatal_crashes_dec2024.xlsx'

# Read Sheet 1: main fatal crashes data
df_crashes_main = pd.read_excel(crashes_file, sheet_name=1, skiprows=4)

# File 2: bitre_fatalities_dec2024.xlsx
fatalities_file = base_path + 'bitre_fatalities_dec2024.xlsx'

# Read Sheet 2 
df_fatalities = pd.read_excel(fatalities_file, sheet_name=1, skiprows=4)

# File 3: LGA (count of dwellings).csv
dwellings_file = base_path + 'LGA (count of dwellings).csv'

# Read first 2 rows, from line 12 to 568 (skip 11 rows and read 557 rows)
df_dwellings = pd.read_csv(dwellings_file, skiprows=10, nrows=558, usecols=[0, 1])
df_dwellings = df_dwellings.rename(columns={'Unnamed: 1': 'count'})

# File 4: Population estimates by LGA and other areas
population_file = base_path + 'Population estimates by LGA, Significant Urban Area, Remoteness Area, Commonwealth Electoral Division and State Electoral Division, 2001 to 2023.xlsx'

# Read Table 1. Estimated resident population, Local Government Areas, Australia
df_population_lga = pd.read_excel(population_file, sheet_name=1, skiprows=6, nrows=548)

# Read Table 3. Estimated resident population, Remoteness Areas, Australia
df_population_remote = pd.read_excel(population_file, sheet_name=3, skiprows=50, nrows=6).iloc[:, 1:]

# Print out the shape of each DataFrame to confirm successful loading
print("DataFrames loaded:")
print("df_crashes_main:", df_crashes_main.shape)
print("df_fatalities:", df_fatalities.shape)
print("df_dwellings:", df_dwellings.shape)
print("df_population_lga:", df_population_lga.shape)
print("df_population_remote:", df_population_remote.shape)

DataFrames loaded:
df_crashes_main: (51284, 20)
df_fatalities: (56874, 23)
df_dwellings: (558, 2)
df_population_lga: (548, 25)
df_population_remote: (6, 24)


Look at the dataframes we got.

In [3]:
df_crashes_main.head()

,Crash ID,State,Month,Year,Dayweek,Time,Crash Type,Number Fatalities,Bus \nInvolvement,Heavy Rigid Truck Involvement,Articulated Truck Involvement,Speed Limit,National Remoteness Areas,SA4 Name 2021,National LGA Name 2021,National Road Type,Christmas Period,Easter Period,Day of week,Time of Day
0,20241115,NSW,12,2024,Friday,04:00:00,Single,1,No,No,No,100,Inner Regional Australia,Riverina,Wagga Wagga,Arterial Road,Yes,No,Weekday,Night
1,20241125,NSW,12,2024,Friday,06:15:00,Single,1,No,No,No,80,Inner Regional Australia,Sydney - Baulkham Hills and Hawkesbury,Hawkesbury,Local Road,No,No,Weekday,Day
2,20246013,Tas,12,2024,Friday,09:43:00,Multiple,1,No,No,No,50,Inner Regional Australia,Launceston and North East,Northern Midlands,Local Road,Yes,No,Weekday,Day
3,20241002,NSW,12,2024,Friday,10:35:00,Multiple,1,No,No,No,100,Outer Regional Australia,New England and North West,Armidale Regional,National or State Highway,No,No,Weekday,Day
4,20242261,Vic,12,2024,Friday,11:30:00,Multiple,1,-9,-9,-9,-9,Unknown,NaN,NaN,Undetermined,No,No,Weekday,Day


In [4]:
df_fatalities.head()

,Crash ID,State,Month,Year,Dayweek,Time,Crash Type,Bus Involvement,Heavy Rigid Truck Involvement,Articulated Truck Involvement,...,Age,National Remoteness Areas,SA4 Name 2021,National LGA Name 2021,National Road Type,Christmas Period,Easter Period,Age Group,Day of week,Time of day
0,20241115,NSW,12,2024,Friday,04:00:00,Single,No,No,No,...,74,Inner Regional Australia,Riverina,Wagga Wagga,Arterial Road,Yes,No,65_to_74,Weekday,Night
1,20241125,NSW,12,2024,Friday,06:15:00,Single,No,No,No,...,19,Inner Regional Australia,Sydney - Baulkham Hills and Hawkesbury,Hawkesbury,Local Road,No,No,17_to_25,Weekday,Day
2,20246013,Tas,12,2024,Friday,09:43:00,Multiple,No,No,No,...,33,Inner Regional Australia,Launceston and North East,Northern Midlands,Local Road,Yes,No,26_to_39,Weekday,Day
3,20241002,NSW,12,2024,Friday,10:35:00,Multiple,No,No,No,...,32,Outer Regional Australia,New England and North West,Armidale Regional,National or State Highway,No,No,26_to_39,Weekday,Day
4,20242261,Vic,12,2024,Friday,11:30:00,Multiple,-9,-9,-9,...,62,Unknown,NaN,NaN,Undetermined,No,No,40_to_64,Weekday,Day


In [5]:
df_dwellings.head()

,LGA (EN),count
0,Albury,25430.0
1,Armidale Regional,12955.0
2,Ballina,20889.0
3,Balranald,1091.0
4,Bathurst Regional,18458.0


In [6]:
# Convert 'count' column to integer with nullable Int64 type (preserves NaN)
df_dwellings['count'] = pd.to_numeric(df_dwellings['count'], errors='coerce').astype(pd.Int64Dtype())
df_dwellings.head()

,LGA (EN),count
0,Albury,25430
1,Armidale Regional,12955
2,Ballina,20889
3,Balranald,1091
4,Bathurst Regional,18458


In [7]:
# Adjust column names for df_population_lga
new_years = list(map(str, range(2001, 2024)))
df_population_lga.columns = ['LGA code', 'Local Government Area'] + new_years
df_population_lga.head()

,LGA code,Local Government Area,2001,2002,2003,2004,2005,2006,2007,2008,...,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
0,10050.0,Albury,45265,45816,46180,46505,47004,47566,48140,48518,...,50990,51486,52171,53056,53922,54657,55466,56067,56665,57517
1,10180.0,Armidale,27906,27774,27610,27410,27350,27377,27468,27788,...,29015,29160,29310,29519,29631,29701,29600,29332,29361,29594
2,10250.0,Ballina,37856,38417,38870,39120,39305,39537,39824,40020,...,41881,42336,42993,43652,44385,44997,45663,46196,46849,47279
3,10300.0,Balranald,2751,2703,2661,2596,2545,2507,2473,2433,...,2376,2364,2330,2338,2308,2287,2257,2208,2210,2202
4,10470.0,Bathurst,35504,35831,36084,36245,36547,36916,37272,37904,...,41157,41694,42244,42583,42882,43207,43444,43674,44110,44612


We only used the line 52-57 "National Remoteness Areas" in the population data. So we need to put column names on df_population_remote.

In [8]:
# Create a list of new column names:
# First column: "Remoteness Area", followed by years from 2001 to 2023.
new_columns = ['Remoteness Area'] + [str(year) for year in range(2001, 2024)]

# Assign the new column names to the DataFrame.
df_population_remote.columns = new_columns
df_population_remote.head()

,Remoteness Area,2001,2002,2003,2004,2005,2006,2007,2008,2009,...,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
0,Major Cities of Australia,13398632,13576969,13758296,13926699,14112583,14323547,14627852,14958910,15305349,...,16757374,17060142,17392876,17726860,18028187,18329669,18571518,18542637,18793212,19343743
1,Inner Regional Australia,3571477,3607774,3646934,3685321,3729153,3773105,3823935,3881708,3944521,...,4190104,4229215,4272829,4330441,4391150,4451987,4510529,4564349,4623765,4683975
2,Outer Regional Australia,1819652,1826869,1833501,1840092,1853962,1871586,1888740,1913862,1939221,...,2015440,2022084,2027519,2039982,2049880,2059373,2072522,2082855,2099419,2119123
3,Remote Australia,290537,290491,290607,290118,290339,291387,292480,295488,298758,...,303587,300093,295847,295564,295842,296682,298482,299976,301750,304013
4,Very Remote Australia,194403,193107,191399,190492,190807,191341,194615,199231,203804,...,209181,204461,201836,199741,198199,197115,196197,195595,196253,198024


## Transform

Keep population rows where Year is between 2020 and 2023. Crash and fatal data is not changed.

In [9]:
# Keep only rows where Year is between 2020 and 2023
df_population_remote = df_population_remote[['Remoteness Area', '2019', '2020', '2021', '2022', '2023']]
df_population_lga = df_population_lga[['LGA code', 'Local Government Area', '2019', '2020', '2021', '2022', '2023']]
print("df_population_remote:", df_population_remote.shape)
print("df_population_lga:", df_population_lga.shape)

df_population_remote: (6, 6)
df_population_lga: (548, 7)


### Handle Invalid Values

Identify rows with invalid or missing values to assess data quality before cleaning.

In [10]:
import numpy as np

# Define a list of invalid values (case-insensitive)
invalid_values = ['-9', 'unknown', 'undetermined', '']

# Function to check whether a cell is invalid
def is_invalid(cell):
    if pd.isna(cell):  # Check for NaN
        return True
    if isinstance(cell, str):  # If string, check against keywords
        return cell.strip().lower() in invalid_values
    if isinstance(cell, (int, float)) and str(int(cell)) == '-9':  # Check numeric -9
        return True
    return False

# Apply to df_crashes_main
invalid_rows_crashes = df_crashes_main.map(is_invalid).any(axis=1)
invalid_count_crashes = invalid_rows_crashes.sum()

# Apply to df_fatalities
invalid_rows_fatalities = df_fatalities.map(is_invalid).any(axis=1)
invalid_count_fatalities = invalid_rows_fatalities.sum()

# Print results
print(f"Number of rows with invalid values in df_crashes_main: {invalid_count_crashes}")
print(f"Number of rows with invalid values in df_fatalities: {invalid_count_fatalities}")

Number of rows with invalid values in df_crashes_main: 41268
Number of rows with invalid values in df_fatalities: 46337


Now we know the number of rows with invalid values is more than 10% of the whole dataset, we can't just drop them. Instead, we replace these values with NaN to keep the structure and handle them appropriately later, such as using them in filtering, grouping, or imputing. 

In [11]:
# Define case-insensitive list of invalid values
invalid_values = ['-9', 'unknown', 'undetermined', '', ' ']

# Function to convert any matching (case-insensitive) string to NaN
def clean_cell(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, str) and x.strip().lower() in invalid_values:
        return np.nan
    if isinstance(x, (int, float)) and str(int(x)) == '-9':
        return np.nan
    return x

# Apply to both DataFrames
df_crashes_main = df_crashes_main.map(clean_cell)
df_fatalities = df_fatalities.map(clean_cell)

df_crashes_main.head(10)

,Crash ID,State,Month,Year,Dayweek,Time,Crash Type,Number Fatalities,Bus \nInvolvement,Heavy Rigid Truck Involvement,Articulated Truck Involvement,Speed Limit,National Remoteness Areas,SA4 Name 2021,National LGA Name 2021,National Road Type,Christmas Period,Easter Period,Day of week,Time of Day
0,20241115,NSW,12,2024,Friday,04:00:00,Single,1,No,No,No,100,Inner Regional Australia,Riverina,Wagga Wagga,Arterial Road,Yes,No,Weekday,Night
1,20241125,NSW,12,2024,Friday,06:15:00,Single,1,No,No,No,80,Inner Regional Australia,Sydney - Baulkham Hills and Hawkesbury,Hawkesbury,Local Road,No,No,Weekday,Day
2,20246013,Tas,12,2024,Friday,09:43:00,Multiple,1,No,No,No,50,Inner Regional Australia,Launceston and North East,Northern Midlands,Local Road,Yes,No,Weekday,Day
3,20241002,NSW,12,2024,Friday,10:35:00,Multiple,1,No,No,No,100,Outer Regional Australia,New England and North West,Armidale Regional,National or State Highway,No,No,Weekday,Day
4,20242261,Vic,12,2024,Friday,11:30:00,Multiple,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,No,Weekday,Day
5,20243185,Qld,12,2024,Friday,13:00:00,Multiple,1,No,No,No,100,Inner Regional Australia,Toowoomba,Lockyer Valley,National or State Highway,No,No,Weekday,Day
6,20244016,SA,12,2024,Friday,13:39:00,Single,1,No,No,No,100,Outer Regional Australia,Barossa - Yorke - Mid North,Wakefield,Sub-arterial Road,Yes,No,Weekday,Day
7,20245001,WA,12,2024,Friday,17:15:00,Single,1,No,No,No,90,NaN,NaN,NaN,NaN,No,No,Weekday,Day
8,20243168,Qld,12,2024,Friday,19:00:00,Multiple,1,No,No,No,60,Inner Regional Australia,Darling Downs - Maranoa,Southern Downs,Local Road,No,No,Weekend,Night
9,20246003,Tas,12,2024,Friday,19:30:00,Multiple,1,No,No,No,50,Inner Regional Australia,Hobart,Brighton,Local Road,No,No,Weekend,Night


### Handle Numerical Values

I think the current "Time of Day" is a little simple. To support more meaningful time-based analysis of fatal crash and fatality data, each time entry is classified into a more detailed time-of-day category.

Late Night: 00:00 – 04:59

Early Morning: 05:00 – 06:59

Morning Peak: 07:00 – 09:00

Morning After Peak: 09:01 – 11:59

Noon: 12:00 – 13:29

Afternoon: 13:30 – 14:59

School Dismissal: 15:00 – 15:59

Evening Peak: 16:00 – 18:30

Night: 18:31 – 23:59

Morning peak hours (07:00–09:00) and Evening peak hours (16:00–18:30) are based on definitions used by Australian transport authorities. The school dismissal period (15:00–16:00) reflects the standard end time of primary and secondary schools in Australia, which is typically around 3pm.

In [12]:
import numpy as np

# Function to categorize time into defined periods
def get_time_period(t):
    if pd.isna(t):
        return np.nan
    try:
        h, m, s = map(int, str(t).split(':'))
        total_minutes = h * 60 + m
    except:
        return np.nan

    if total_minutes < 300:
        return 'Late Night'
    elif total_minutes < 420:
        return 'Early Morning'
    elif total_minutes <= 540:
        return 'Morning Peak'
    elif total_minutes <= 719:
        return 'Morning After Peak'
    elif total_minutes <= 809:
        return 'Noon'
    elif total_minutes <= 899:
        return 'Afternoon'
    elif total_minutes <= 959:
        return 'School Dismissal'
    elif total_minutes <= 1110:
        return 'Evening Peak'
    else:
        return 'Night'

# Apply to both DataFrames
df_crashes_main['Time Period'] = df_crashes_main['Time'].apply(get_time_period)
df_fatalities['Time Period'] = df_fatalities['Time'].apply(get_time_period)

# Drop 'Time of Day' column from both DataFrames (ignore error if column doesn't exist)
df_crashes_main = df_crashes_main.drop(columns=['Time of Day'], errors='ignore')
df_fatalities = df_fatalities.drop(columns=['Time of Day'], errors='ignore')

df_crashes_main.head()

,Crash ID,State,Month,Year,Dayweek,Time,Crash Type,Number Fatalities,Bus \nInvolvement,Heavy Rigid Truck Involvement,Articulated Truck Involvement,Speed Limit,National Remoteness Areas,SA4 Name 2021,National LGA Name 2021,National Road Type,Christmas Period,Easter Period,Day of week,Time Period
0,20241115,NSW,12,2024,Friday,04:00:00,Single,1,No,No,No,100,Inner Regional Australia,Riverina,Wagga Wagga,Arterial Road,Yes,No,Weekday,Late Night
1,20241125,NSW,12,2024,Friday,06:15:00,Single,1,No,No,No,80,Inner Regional Australia,Sydney - Baulkham Hills and Hawkesbury,Hawkesbury,Local Road,No,No,Weekday,Early Morning
2,20246013,Tas,12,2024,Friday,09:43:00,Multiple,1,No,No,No,50,Inner Regional Australia,Launceston and North East,Northern Midlands,Local Road,Yes,No,Weekday,Morning After Peak
3,20241002,NSW,12,2024,Friday,10:35:00,Multiple,1,No,No,No,100,Outer Regional Australia,New England and North West,Armidale Regional,National or State Highway,No,No,Weekday,Morning After Peak
4,20242261,Vic,12,2024,Friday,11:30:00,Multiple,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,No,Weekday,Morning After Peak


In most Australian states and territories (WA, VIC, NSW, TAS, ACT, SA, and QLD), the default speed limit for built-up areas is 50 km/h. In the Northern Territory (NT), the default built-up limit is 60 km/h. Based on this, speed values were classified as follows:

For NT:
low: speed < 60
medium: 60 ≤ speed ≤ 80
high: speed > 80

For all other states:
low: speed < 50
medium: 50 ≤ speed ≤ 80
high: speed > 80


To ensure consistency in data types and simplify numeric analysis, we manually converted the textual value "<40" found in the Speed Limit column to a numerical value of 30. This decision was made because only 8 rows is "<40" in this dataset, and treating them as 30 allows us to preserve an INTEGER data type for the entire column.

By doing so, we avoid having to treat Speed Limit as a TEXT field in the data warehouse schema, which would otherwise complicate filtering, grouping, and aggregation operations during analysis.

In [13]:
def get_speed_category(row):
    speed = row.get('Speed Limit')
    state = row.get('State')

    # Skip rows with missing or invalid speed
    if pd.isna(speed) or pd.isna(state):
        return np.nan
    try:
        speed = float(speed)
    except:
        return np.nan
    
    df_crashes_main['Speed Limit'] = df_crashes_main['Speed Limit'].replace('<40', 30)
    df_fatalities['Speed Limit'] = df_fatalities['Speed Limit'].replace('<40', 30)

    df_crashes_main['Speed Limit'] = pd.to_numeric(df_crashes_main['Speed Limit'], errors='coerce').astype(pd.Int64Dtype())
    df_fatalities['Speed Limit'] = pd.to_numeric(df_fatalities['Speed Limit'], errors='coerce').astype(pd.Int64Dtype())


    # Apply NT-specific rules
    if state.strip().upper() == 'NT':
        if speed < 60:
            return 'low'
        elif speed <= 80:
            return 'medium'
        else:
            return 'high'
    else:
        # All other states
        if speed < 50:
            return 'low'
        elif speed <= 80:
            return 'medium'
        else:
            return 'high'

# Apply to both DataFrames
df_crashes_main['Speed Category'] = df_crashes_main.apply(get_speed_category, axis=1)
df_fatalities['Speed Category'] = df_fatalities.apply(get_speed_category, axis=1)

df_crashes_main.head()

/var/folders/59/k1t7jcln0kz605z8w_ytj9t00000gn/T/ipykernel_77920/43280949.py:13: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_crashes_main['Speed Limit'] = df_crashes_main['Speed Limit'].replace('<40', 30)
/var/folders/59/k1t7jcln0kz605z8w_ytj9t00000gn/T/ipykernel_77920/43280949.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_fatalities['Speed Limit'] = df_fatalities['Speed Limit'].replace('<40', 30)


,Crash ID,State,Month,Year,Dayweek,Time,Crash Type,Number Fatalities,Bus \nInvolvement,Heavy Rigid Truck Involvement,...,Speed Limit,National Remoteness Areas,SA4 Name 2021,National LGA Name 2021,National Road Type,Christmas Period,Easter Period,Day of week,Time Period,Speed Category
0,20241115,NSW,12,2024,Friday,04:00:00,Single,1,No,No,...,100,Inner Regional Australia,Riverina,Wagga Wagga,Arterial Road,Yes,No,Weekday,Late Night,high
1,20241125,NSW,12,2024,Friday,06:15:00,Single,1,No,No,...,80,Inner Regional Australia,Sydney - Baulkham Hills and Hawkesbury,Hawkesbury,Local Road,No,No,Weekday,Early Morning,medium
2,20246013,Tas,12,2024,Friday,09:43:00,Multiple,1,No,No,...,50,Inner Regional Australia,Launceston and North East,Northern Midlands,Local Road,Yes,No,Weekday,Morning After Peak,medium
3,20241002,NSW,12,2024,Friday,10:35:00,Multiple,1,No,No,...,100,Outer Regional Australia,New England and North West,Armidale Regional,National or State Highway,No,No,Weekday,Morning After Peak,high
4,20242261,Vic,12,2024,Friday,11:30:00,Multiple,1,NaN,NaN,...,<NA>,NaN,NaN,NaN,NaN,No,No,Weekday,Morning After Peak,NaN


In [14]:
df_fatalities.head()

,Crash ID,State,Month,Year,Dayweek,Time,Crash Type,Bus Involvement,Heavy Rigid Truck Involvement,Articulated Truck Involvement,...,SA4 Name 2021,National LGA Name 2021,National Road Type,Christmas Period,Easter Period,Age Group,Day of week,Time of day,Time Period,Speed Category
0,20241115,NSW,12,2024,Friday,04:00:00,Single,No,No,No,...,Riverina,Wagga Wagga,Arterial Road,Yes,No,65_to_74,Weekday,Night,Late Night,high
1,20241125,NSW,12,2024,Friday,06:15:00,Single,No,No,No,...,Sydney - Baulkham Hills and Hawkesbury,Hawkesbury,Local Road,No,No,17_to_25,Weekday,Day,Early Morning,medium
2,20246013,Tas,12,2024,Friday,09:43:00,Multiple,No,No,No,...,Launceston and North East,Northern Midlands,Local Road,Yes,No,26_to_39,Weekday,Day,Morning After Peak,medium
3,20241002,NSW,12,2024,Friday,10:35:00,Multiple,No,No,No,...,New England and North West,Armidale Regional,National or State Highway,No,No,26_to_39,Weekday,Day,Morning After Peak,high
4,20242261,Vic,12,2024,Friday,11:30:00,Multiple,NaN,NaN,NaN,...,NaN,NaN,NaN,No,No,40_to_64,Weekday,Day,Morning After Peak,NaN


In [15]:
print(df_fatalities['Age'].head())

0    74.0
1    19.0
2    33.0
3    32.0
4    62.0
Name: Age, dtype: float64


Change Age type to int

In [16]:
df_fatalities['Age'] = pd.to_numeric(df_fatalities['Age'], errors='coerce').astype('Int64')
print(df_fatalities['Age'].head())

0    74
1    19
2    33
3    32
4    62
Name: Age, dtype: Int64


In [17]:
df_population_remote.head()

,Remoteness Area,2019,2020,2021,2022,2023
0,Major Cities of Australia,18329669,18571518,18542637,18793212,19343743
1,Inner Regional Australia,4451987,4510529,4564349,4623765,4683975
2,Outer Regional Australia,2059373,2072522,2082855,2099419,2119123
3,Remote Australia,296682,298482,299976,301750,304013
4,Very Remote Australia,197115,196197,195595,196253,198024


In [18]:
df_population_lga.head()

,LGA code,Local Government Area,2019,2020,2021,2022,2023
0,10050.0,Albury,54657,55466,56067,56665,57517
1,10180.0,Armidale,29701,29600,29332,29361,29594
2,10250.0,Ballina,44997,45663,46196,46849,47279
3,10300.0,Balranald,2287,2257,2208,2210,2202
4,10470.0,Bathurst,43207,43444,43674,44110,44612


In [19]:
df_dwellings.head()

,LGA (EN),count
0,Albury,25430
1,Armidale Regional,12955
2,Ballina,20889
3,Balranald,1091
4,Bathurst Regional,18458


### Merge 'Number Fatalities' data

Although the df_fatalities dataset already includes almost all the attributes from df_crashes_main, it does not contain the Number Fatalities field.

To address this, we merged the Number Fatalities column from df_crashes_main into df_fatalities using the Crash ID as the key. Since there is a one-to-many relationship between crashes and fatalities (i.e., each crash may involve multiple fatalities), the same Number Fatalities value will be repeated for all fatalities linked to the same crash.

This approach ensures that we retain the granularity of individual fatality records while incorporating a key measure (Number Fatalities) useful for later analysis.

In [20]:
def merge_number_of_fatalities(df_fatalities, df_crashes_main):
    # Select only Crash ID and Number of Fatalities from crash data
    fatalities_count = df_crashes_main[['Crash ID', 'Number Fatalities']]

    # Merge into fatalities data based on Crash ID
    df_merged = df_fatalities.merge(fatalities_count, on='Crash ID', how='left')

    # Add fatalityId as unique auto-incrementing identifier
    df_merged = df_merged.copy()
    df_merged['fatalityId'] = range(1, len(df_merged) + 1)

    return df_merged

# Use the function
df_fatalities_merged = merge_number_of_fatalities(df_fatalities, df_crashes_main)

# Preview the first 20 rows
df_fatalities_merged.head(20)

,Crash ID,State,Month,Year,Dayweek,Time,Crash Type,Bus Involvement,Heavy Rigid Truck Involvement,Articulated Truck Involvement,...,National Road Type,Christmas Period,Easter Period,Age Group,Day of week,Time of day,Time Period,Speed Category,Number Fatalities,fatalityId
0,20241115,NSW,12,2024,Friday,04:00:00,Single,No,No,No,...,Arterial Road,Yes,No,65_to_74,Weekday,Night,Late Night,high,1,1
1,20241125,NSW,12,2024,Friday,06:15:00,Single,No,No,No,...,Local Road,No,No,17_to_25,Weekday,Day,Early Morning,medium,1,2
2,20246013,Tas,12,2024,Friday,09:43:00,Multiple,No,No,No,...,Local Road,Yes,No,26_to_39,Weekday,Day,Morning After Peak,medium,1,3
3,20241002,NSW,12,2024,Friday,10:35:00,Multiple,No,No,No,...,National or State Highway,No,No,26_to_39,Weekday,Day,Morning After Peak,high,1,4
4,20242261,Vic,12,2024,Friday,11:30:00,Multiple,NaN,NaN,NaN,...,NaN,No,No,40_to_64,Weekday,Day,Morning After Peak,NaN,1,5
5,20243185,Qld,12,2024,Friday,13:00:00,Multiple,No,No,No,...,National or State Highway,No,No,40_to_64,Weekday,Day,Noon,high,1,6
6,20244016,SA,12,2024,Friday,13:39:00,Single,No,No,No,...,Sub-arterial Road,Yes,No,40_to_64,Weekday,Day,Afternoon,high,1,7
7,20245001,WA,12,2024,Friday,17:15:00,Single,No,No,No,...,NaN,No,No,65_to_74,Weekday,Day,Evening Peak,high,1,8
8,20243168,Qld,12,2024,Friday,19:00:00,Multiple,No,No,No,...,Local Road,No,No,26_to_39,Weekend,Night,Night,medium,1,9
9,20246003,Tas,12,2024,Friday,19:30:00,Multiple,No,No,No,...,Local Road,No,No,17_to_25,Weekend,Night,Night,medium,1,10


## Load
### Create fact & dimension tables 

In [21]:
def create_dim_date(df):
    # Select relevant columns
    dim_date = df[['Year', 'Month', 'Day of week', 'Christmas Period', 'Easter Period']].copy()

    # Rename columns for consistency
    dim_date = dim_date.rename(columns={
        'Year': 'year',
        'Month': 'month',
        'Day of week': 'day_of_week',
        'Christmas Period': 'is_christmas_period',
        'Easter Period': 'is_easter_period'
    })

    # Convert Yes/No to boolean
    dim_date['is_christmas_period'] = dim_date['is_christmas_period'].str.strip().str.lower() == 'yes'
    dim_date['is_easter_period'] = dim_date['is_easter_period'].str.strip().str.lower() == 'yes'

    # Add auto-incrementing date_id
    dim_date = dim_date.reset_index(drop=True)
    dim_date['date_id'] = dim_date.index + 1

    # Reorder columns
    dim_date = dim_date[['date_id', 'year', 'month', 'day_of_week',
                         'is_christmas_period', 'is_easter_period']]

    return dim_date

dim_date = create_dim_date(df_fatalities_merged)

dim_date.head(10)

,date_id,year,month,day_of_week,is_christmas_period,is_easter_period
0,1,2024,12,Weekday,True,False
1,2,2024,12,Weekday,False,False
2,3,2024,12,Weekday,True,False
3,4,2024,12,Weekday,False,False
4,5,2024,12,Weekday,False,False
5,6,2024,12,Weekday,False,False
6,7,2024,12,Weekday,True,False
7,8,2024,12,Weekday,False,False
8,9,2024,12,Weekend,False,False
9,10,2024,12,Weekend,False,False


In [22]:
def create_dim_time(df):
    # Select and deduplicate relevant columns
    dim_time = df[['Time', 'Time Period']].copy()

    # Rename columns
    dim_time = dim_time.rename(columns={
        'Time': 'time',
        'Time Period': 'time_period'
    })

    # Add auto-incrementing time_id
    dim_time = dim_time.reset_index(drop=True)
    dim_time['time_id'] = dim_time.index + 1

    # Reorder columns
    dim_time = dim_time[['time_id', 'time_period', 'time']]

    return dim_time

dim_time = create_dim_time(df_fatalities_merged)

dim_time.head()

,time_id,time_period,time
0,1,Late Night,04:00:00
1,2,Early Morning,06:15:00
2,3,Morning After Peak,09:43:00
3,4,Morning After Peak,10:35:00
4,5,Morning After Peak,11:30:00


In [23]:
def create_dim_location(df):
    # Select and deduplicate relevant geographic fields
    dim_location = df[['State', 'SA4 Name 2021', 'National LGA Name 2021', 'National Remoteness Areas']].copy()

    # Rename columns
    dim_location = dim_location.rename(columns={
        'State': 'state',
        'SA4 Name 2021': 'sa4_name',
        'National LGA Name 2021': 'lga_name',
        'National Remoteness Areas': 'remoteness_area'
    })

    # Add auto-incrementing location_id
    dim_location = dim_location.reset_index(drop=True)
    dim_location['location_id'] = dim_location.index + 1

    # Reorder columns
    dim_location = dim_location[['location_id', 'state', 'sa4_name', 'lga_name', 'remoteness_area']]

    return dim_location

dim_location = create_dim_location(df_fatalities_merged)
dim_location.head()

,location_id,state,sa4_name,lga_name,remoteness_area
0,1,NSW,Riverina,Wagga Wagga,Inner Regional Australia
1,2,NSW,Sydney - Baulkham Hills and Hawkesbury,Hawkesbury,Inner Regional Australia
2,3,Tas,Launceston and North East,Northern Midlands,Inner Regional Australia
3,4,NSW,New England and North West,Armidale Regional,Outer Regional Australia
4,5,Vic,NaN,NaN,NaN


In [24]:
def create_dim_road_type(df):
    # Select and deduplicate relevant columns
    dim_road_type = df[['National Road Type', 'Speed Limit', 'Speed Category']].copy()

    # Rename columns
    dim_road_type = dim_road_type.rename(columns={
        'National Road Type': 'road_type',
        'Speed Limit': 'speed_limit',
        'Speed Category': 'speed_category'
    })

    # Add auto-incrementing primary key
    dim_road_type = dim_road_type.reset_index(drop=True)
    dim_road_type['road_type_id'] = dim_road_type.index + 1

    # Reorder columns
    dim_road_type = dim_road_type[['road_type_id', 'road_type', 'speed_limit', 'speed_category']]

    return dim_road_type

dim_road_type = create_dim_road_type(df_fatalities_merged)
dim_road_type.head()

,road_type_id,road_type,speed_limit,speed_category
0,1,Arterial Road,100,high
1,2,Local Road,80,medium
2,3,Local Road,50,medium
3,4,National or State Highway,100,high
4,5,NaN,<NA>,NaN


In [25]:
def create_dim_crash_details(df):
    # Select relevant columns
    dim_crash = df[['Crash Type', 'Bus Involvement', 'Heavy Rigid Truck Involvement', 'Articulated Truck Involvement']].copy()

    # Rename columns
    dim_crash = dim_crash.rename(columns={
        'Crash Type': 'crash_type',
        'Bus Involvement': 'bus_involvement',
        'Heavy Rigid Truck Involvement': 'heavy_rigid_truck_involvement',
        'Articulated Truck Involvement': 'articulated_truck_involvement'
    })

    # Convert Yes/No to boolean
    for col in ['bus_involvement', 'heavy_rigid_truck_involvement', 'articulated_truck_involvement']:
        dim_crash[col] = dim_crash[col].str.strip().str.lower() == 'yes'

    # Add auto-incrementing primary key
    dim_crash = dim_crash.reset_index(drop=True)
    dim_crash['crash_details_id'] = dim_crash.index + 1

    # Reorder columns
    dim_crash = dim_crash[['crash_details_id', 'crash_type',
                           'bus_involvement',
                           'heavy_rigid_truck_involvement',
                           'articulated_truck_involvement']]

    return dim_crash

dim_crash_details = create_dim_crash_details(df_fatalities_merged)
dim_crash_details.head()

,crash_details_id,crash_type,bus_involvement,heavy_rigid_truck_involvement,articulated_truck_involvement
0,1,Single,False,False,False
1,2,Single,False,False,False
2,3,Multiple,False,False,False
3,4,Multiple,False,False,False
4,5,Multiple,False,False,False


In [26]:
def create_dim_demographic(df):
    # Select and deduplicate relevant demographic columns
    dim_demo = df[['Age Group', 'Gender', 'Age', 'Road User']].copy()

    # Rename columns
    dim_demo = dim_demo.rename(columns={
        'Age Group': 'age_group',
        'Gender': 'gender',
        'Age': 'age',
        'Road User': 'road_user'
    })

    # Add auto-incrementing primary key
    dim_demo = dim_demo.reset_index(drop=True)
    dim_demo['demographic_id'] = dim_demo.index + 1

    # Reorder columns
    dim_demo = dim_demo[['demographic_id', 'gender', 'age_group', 'age', 'road_user']]

    return dim_demo

dim_demographic = create_dim_demographic(df_fatalities_merged)
dim_demographic.head()

,demographic_id,gender,age_group,age,road_user
0,1,Male,65_to_74,74,Driver
1,2,Female,17_to_25,19,Driver
2,3,Female,26_to_39,33,Driver
3,4,Female,26_to_39,32,Driver
4,5,Male,40_to_64,62,Passenger


In [27]:
def create_dim_remoteness(df):
    # Rename columns for consistency
    dim_remote = df.rename(columns={
        'Remoteness Area': 'remoteness_area',
        '2020': 'population_2020',
        '2021': 'population_2021',
        '2022': 'population_2022',
        '2023': 'population_2023'
    })

    # Add auto-incrementing remoteness_id
    dim_remote = dim_remote.reset_index(drop=True)
    dim_remote['remoteness_id'] = dim_remote.index + 1

    # Reorder columns
    dim_remote = dim_remote[['remoteness_id', 'remoteness_area',
                             'population_2020', 'population_2021',
                             'population_2022', 'population_2023']]

    return dim_remote

dim_remoteness = create_dim_remoteness(df_population_remote)
dim_remoteness.head()

,remoteness_id,remoteness_area,population_2020,population_2021,population_2022,population_2023
0,1,Major Cities of Australia,18571518,18542637,18793212,19343743
1,2,Inner Regional Australia,4510529,4564349,4623765,4683975
2,3,Outer Regional Australia,2072522,2082855,2099419,2119123
3,4,Remote Australia,298482,299976,301750,304013
4,5,Very Remote Australia,196197,195595,196253,198024


In [28]:
def create_dim_lga(df_dwellings, df_population_lga):
    # Rename columns for consistency
    df_dwellings = df_dwellings.rename(columns={'LGA (EN)': 'lga_name', 'count': 'dwelling_count'})
    df_population_lga = df_population_lga.rename(columns={
        'Local Government Area': 'lga_name',
        'LGA code': 'lga_code',
        '2020': 'population_2020',
        '2021': 'population_2021',
        '2022': 'population_2022',
        '2023': 'population_2023'
    })

    # Merge on original LGA name
    df_lga = df_population_lga.merge(df_dwellings, on='lga_name', how='left')

    # Fill missing dwelling count with 0 if any
    df_lga['dwelling_count'] = df_lga['dwelling_count'].fillna(0).astype(int)

    # Add auto-incrementing primary key
    df_lga = df_lga.reset_index(drop=True)
    df_lga['lga_id'] = df_lga.index + 1

    # Reorder columns
    df_lga = df_lga[['lga_id', 'lga_name', 'dwelling_count',
                     'population_2020', 'population_2021',
                     'population_2022', 'population_2023',
                     'lga_code']]

    return df_lga

dim_lga = create_dim_lga(df_dwellings, df_population_lga)
dim_lga.head()

,lga_id,lga_name,dwelling_count,population_2020,population_2021,population_2022,population_2023,lga_code
0,1,Albury,25430,55466,56067,56665,57517,10050.0
1,2,Armidale,0,29600,29332,29361,29594,10180.0
2,3,Ballina,20889,45663,46196,46849,47279,10250.0
3,4,Balranald,1091,2257,2208,2210,2202,10300.0
4,5,Bathurst,0,43444,43674,44110,44612,10470.0


To build the initial version of the fact_crash_fatalities table, we know that first 6 dimension tables (apart from remoteness and lga) were constructed directly from df_fatalities_merged in a row-wise manner.

Based on this alignment, we directly assigned dimension foreign keys (IDs) in the fact table by matching row indices. This avoids costly joins and ensures fast generation of the core structure of the fact table.

In [29]:
def create_fact_crash_fatalities_partial(df_fatalities_merged):
    fact = pd.DataFrame()

    # Use fatalityId as the primary key
    fact['fatality_id'] = df_fatalities_merged['fatalityId']

    # Extract crash_id and number_of_fatalities
    fact['crash_id'] = df_fatalities_merged['Crash ID']
    fact['number_of_fatalities'] = df_fatalities_merged['Number Fatalities']

    # Assume all IDs (date_id, time_id, etc.) match by row index
    fact['date_id'] = df_fatalities_merged.index + 1
    fact['time_id'] = df_fatalities_merged.index + 1
    fact['location_id'] = df_fatalities_merged.index + 1
    fact['road_type_id'] = df_fatalities_merged.index + 1
    fact['crash_details_id'] = df_fatalities_merged.index + 1
    fact['demographic_id'] = df_fatalities_merged.index + 1

    return fact

fact_partial = create_fact_crash_fatalities_partial(df_fatalities_merged)
fact_partial.head()

,fatality_id,crash_id,number_of_fatalities,date_id,time_id,location_id,road_type_id,crash_details_id,demographic_id
0,1,20241115,1,1,1,1,1,1,1
1,2,20241125,1,2,2,2,2,2,2
2,3,20246013,1,3,3,3,3,3,3
3,4,20241002,1,4,4,4,4,4,4
4,5,20242261,1,5,5,5,5,5,5


The field remoteness_id in the fact table is derived by linking the National Remoteness Areas column in the fatalities dataset to the remoteness_area field in the dim_remoteness dimension table.

This is a one-to-many relationship: each remoteness_area in dim_remoteness represents a broader classification, and many individual fatality records can fall under the same category.

By using a dictionary-based mapping (map), we efficiently assign the appropriate remoteness_id to each record in the fact table, enabling future aggregation and filtering based on regional remoteness levels.

In [30]:
def add_remoteness_id(df_fact_partial, df_fatalities_merged, dim_remoteness):
    # Create a mapping from remoteness category to its ID
    remoteness_map = dict(zip(dim_remoteness['remoteness_area'], dim_remoteness['remoteness_id']))
    
    # Map remoteness_id based on 'National Remoteness Areas' in fatalities dataset
    df_fact_partial['remoteness_id'] = df_fatalities_merged['National Remoteness Areas'].map(remoteness_map)

    # Convert to nullable integer type
    df_fact_partial['remoteness_id'] = df_fact_partial['remoteness_id'].astype('Int64')
    
    return df_fact_partial

fact_partial = add_remoteness_id(fact_partial, df_fatalities_merged, dim_remoteness)
fact_partial.head()

,fatality_id,crash_id,number_of_fatalities,date_id,time_id,location_id,road_type_id,crash_details_id,demographic_id,remoteness_id
0,1,20241115,1,1,1,1,1,1,1,2
1,2,20241125,1,2,2,2,2,2,2,2
2,3,20246013,1,3,3,3,3,3,3,2
3,4,20241002,1,4,4,4,4,4,4,3
4,5,20242261,1,5,5,5,5,5,5,<NA>


The lga_id in the fact table is linked using the National LGA Name 2021 column from the fatalities dataset and the lga_name field in the dim_lga dimension table.

In [31]:
def add_lga_id(df_fact_partial, df_fatalities_merged, dim_lga):
    # Create a mapping from LGA name to its ID
    lga_map = dict(zip(dim_lga['lga_name'], dim_lga['lga_id']))

    # Map lga_id based on 'National LGA Name 2021' in fatalities dataset
    df_fact_partial['lga_id'] = df_fatalities_merged['National LGA Name 2021'].map(lga_map)

    # Convert to nullable integer type
    df_fact_partial['lga_id'] = df_fact_partial['lga_id'].astype('Int64')

    return df_fact_partial

fact_partial = add_lga_id(fact_partial, df_fatalities_merged, dim_lga)
fact_partial.head()

,fatality_id,crash_id,number_of_fatalities,date_id,time_id,location_id,road_type_id,crash_details_id,demographic_id,remoteness_id,lga_id
0,1,20241115,1,1,1,1,1,1,1,2,115
1,2,20241125,1,2,2,2,2,2,2,2,52
2,3,20246013,1,3,3,3,3,3,3,2,520
3,4,20241002,1,4,4,4,4,4,4,3,<NA>
4,5,20242261,1,5,5,5,5,5,5,<NA>,<NA>


Finalize fact table

In [32]:
def finalize_fact_table(fact_crash_fatalities):
    # Rename columns to standard format
    fact_crash_fatalities = fact_crash_fatalities.rename(columns={
        'fatalityId': 'fatality_id',
        'Crash ID': 'crash_id',
        'Number Fatalities': 'number_of_fatalities'
    })
    
    # Replace blank strings with NaN
    fact_crash_fatalities = fact_crash_fatalities.map(
        lambda x: np.nan if isinstance(x, str) and x.strip() == '' else x
    )

    # Convert float columns to nullable Int64
    for col in fact_crash_fatalities.columns:
        if pd.api.types.is_float_dtype(fact_crash_fatalities[col]):
            fact_crash_fatalities[col] = fact_crash_fatalities[col].astype('Int64')

    return fact_crash_fatalities

fact_crash_fatalities = finalize_fact_table(fact_partial)
fact_crash_fatalities.head(10)

,fatality_id,crash_id,number_of_fatalities,date_id,time_id,location_id,road_type_id,crash_details_id,demographic_id,remoteness_id,lga_id
0,1,20241115,1,1,1,1,1,1,1,2,115
1,2,20241125,1,2,2,2,2,2,2,2,52
2,3,20246013,1,3,3,3,3,3,3,2,520
3,4,20241002,1,4,4,4,4,4,4,3,<NA>
4,5,20242261,1,5,5,5,5,5,5,<NA>,<NA>
5,6,20243185,1,6,6,6,6,6,6,2,248
6,7,20244016,1,7,7,7,7,7,7,3,350
7,8,20245001,1,8,8,8,8,8,8,<NA>,<NA>
8,9,20243168,1,9,9,9,9,9,9,2,274
9,10,20246003,1,10,10,10,10,10,10,2,499


All tables exported to CSV

In [33]:
def export_all_tables_to_csv(
    fact_crash_fatalities,
    dim_date,
    dim_time,
    dim_location,
    dim_road_type,
    dim_crash_details,
    dim_demographic,
    dim_remoteness,
    dim_lga,
    output_dir='./'
):
    # Export each table
    fact_crash_fatalities.to_csv(f"{output_dir}fact_crash_fatalities.csv", index=False)
    dim_date.to_csv(f"{output_dir}dim_date.csv", index=False)
    dim_time.to_csv(f"{output_dir}dim_time.csv", index=False)
    dim_location.to_csv(f"{output_dir}dim_location.csv", index=False)
    dim_road_type.to_csv(f"{output_dir}dim_road_type.csv", index=False)
    dim_crash_details.to_csv(f"{output_dir}dim_crash_details.csv", index=False)
    dim_demographic.to_csv(f"{output_dir}dim_demographic.csv", index=False)
    dim_remoteness.to_csv(f"{output_dir}dim_remoteness.csv", index=False)
    dim_lga.to_csv(f"{output_dir}dim_lga.csv", index=False)

    print("All tables exported to CSV successfully.")

export_all_tables_to_csv(
    fact_crash_fatalities,
    dim_date,
    dim_time,
    dim_location,
    dim_road_type,
    dim_crash_details,
    dim_demographic,
    dim_remoteness,
    dim_lga
)

All tables exported to CSV successfully.
